In [ ]:
import numpy as np
import random
from collections import Counter

def generate_traffic_clustered(Nrack_, Hosts_p_rack_, load_, time_, Gbps_rate_, Nactive_, workload_, network_, seedValue_):
    print(f'Running Python function with arguments: {Nrack_}, {Hosts_p_rack_}, {load_}, {time_}, {Gbps_rate_}, {Nactive_}, {workload_}, {network_}, {seedValue_}')
    
    
    np.random.seed(seedValue_)
    random.seed(seedValue_) 

    Nrack = Nrack_
    Hosts_p_rack = Hosts_p_rack_
    H = Nrack * Hosts_p_rack  # number of hosts
    loadfrac0 = load_  # fraction of theoretically possible load
    
    totaltime = time_  # seconds
    rate = Gbps_rate_
    linkrate = rate * 10e8 / 8  # bytes per second
    Nactive = Nactive_
    
    
    filename = f'{network_}/{workload_}_{100 * loadfrac0:.2f}percLoad_{int(totaltime)}sec_{Nrack}N_{Hosts_p_rack}hpr_{H}hosts_{rate}Gbps_{Nactive:.2f}Nactive_seed={seedValue_}.htsim'

    
    H_active = int(np.ceil(H * Nactive))
    print(f'H_active = {H_active}')
    
    Ncons = H_active * (H_active - Hosts_p_rack)  # number of possible connections
    srcdst = np.zeros((Ncons, 2), dtype=int)
    cnt = 0
    cdf = 0
    tmcdf = np.zeros(Ncons)


    # Initialize the probability list
    probabilities = []

    for a in range(H_active):  # sources
        for b in range(H_active):  # destinations
            if a // Hosts_p_rack != b // Hosts_p_rack:
                
                # Store the source-destination pair
                srcdst[cnt] = [a, b]
                probabilities.append(1.0)
                cnt += 1


    # Convert the list to a numpy array for further processing
    probabilities = np.array(probabilities)

    # Normalize the probabilities to ensure they sum to 1
    probabilities /= probabilities.sum()

    # Calculate the cumulative distribution function (CDF) using np.cumsum
    tmcdf = np.cumsum(probabilities)

    print(f"len(srcdst) = {len(srcdst)}")
    print(f"tmcdf = {tmcdf}") 
    print(f"len tmcdf = {len(tmcdf)}")

    # Load flow size distribution from CSV
    if workload_ == 'DM':
        flowdis_data = np.loadtxt('../_flow_dis/DM.csv', delimiter=',')
    elif workload_ == 'HD':
        flowdis_data = np.loadtxt('../_flow_dis/HD.csv', delimiter=',')
    elif workload_ == 'WS':
        flowdis_data = np.loadtxt('../_flow_dis/WS.csv', delimiter=',')
    else:
        raise ValueError('Unknown workload specified')

    flowsize = flowdis_data[:, 0]
    flowcdf = flowdis_data[:, 1]

    print(f"data = {flowdis_data} {type(flowdis_data)}")
    print(f"flowsize = {flowsize}")
    print(f"flowcdf = {flowcdf}")

    
    avg_flowsize = np.sum(flowsize[1:] * np.diff(flowcdf))  # bytes/flow

    lambda_host_max = linkrate / avg_flowsize  # flows/second per host
    lambda_host = loadfrac0 * lambda_host_max  # flows/second for each host

    lambda_network = H_active * lambda_host  # flows/second for the entire network

    nflows_est = int(np.ceil(lambda_network * totaltime))
    flowmat1 = np.zeros((nflows_est, 4), dtype=np.int64)

    print('Getting PRIO flow start times...')
    crt_time = 0
    cnt = 0
    while crt_time < totaltime:
        next_time = -np.log(1 - np.random.rand()) / lambda_network
        crt_time += next_time
        if cnt >= flowmat1.shape[0]:
            flowmat1 = np.vstack([flowmat1, np.zeros((flowmat1.shape[0], 4), dtype=np.int64)])
        flowmat1[cnt, 3] = int(crt_time * 1e9)  # nanoseconds
        cnt += 1

    flowmat1 = flowmat1[:cnt]

    # ind = np.where(flowmat1[:, 3] > totaltime * 1e9)[0]
    # if len(ind) > 0:
    #     flowmat1[ind[0]:, :] = 69  # zero out flows beyond the total time

    print('Getting PRIO flow sizes...')
    randvect = np.random.rand(flowmat1.shape[0])
    indices = np.searchsorted(flowcdf, randvect)
    flowmat1[:, 2] = flowsize[indices]

    print('Getting PRIO flow sources & destinations...')
    randvect = np.random.rand(flowmat1.shape[0])
    indices = np.searchsorted(tmcdf, randvect)
    print(f"indices = {indices}")
    flowmat1[:, 0:2] = srcdst[indices]

    actual_load_frac = np.sum(flowmat1[:, 2]) / (totaltime * H * linkrate)
    print(f'\n\nSpecified fraction of capacity = {loadfrac0:.3f}')
    print(f'Actual fraction of capacity = {actual_load_frac:.3f}\n')

    # Write the flowmat1 to the file in the appropriate format
    write_to_htsim_file(flowmat1, filename)

def write_to_htsim_file(flowmat, filename):
    with open(filename, 'w') as f:
        for idx, row in enumerate(flowmat):
            if idx == len(flowmat)-1:
                f.write(f"{row[0]} {row[1]} {row[2]} {row[3]}")
            else:
                f.write(f"{row[0]} {row[1]} {row[2]} {row[3]}\n")
    print(f"Data written to {filename}")


In [ ]:
Nrack_ = 108
Hosts_p_rack_ = 6
load_ = 0.06
time_ = 10.001
Gbps_rate_ = 40
Nactive_ = 1
workload_ = "DM"
network_ = "opera"
seedValue_ = 2


In [ ]:
load_set = [0.02, 0.05, 0.10, 0.20]
for load_ in load_set:
    generate_traffic_clustered(Nrack_, Hosts_p_rack_, load_, time_, Gbps_rate_, Nactive_, workload_, network_, seedValue_)
